In [87]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

# Package for regridding 
import xesmf as xe 

# Make nice plots with different projections 
import cartopy.crs as ccrs 

# Toolkit for sea level equation
import gravity_toolkit as gravtk

#from scipy import signal              
import importlib

# Functions to make the script more readable. 
import Kyrafunctions as ky

import sys
sys.path.append('../code')
import SeaLevelContrib as slc


importlib.reload(slc)
importlib.reload(ky)

<module 'Kyrafunctions' from 'C:\\Users\\kyrab\\Github\\SLBudget\\notebooks\\Kyrafunctions.py'>

In [71]:
data_path = '../data/GravIs/'
output_path = '../outputs/Gravis_Antarctica/'
path_masks = '/Users/kyrab/Github/SLBudget/data/Masks/'

In [72]:
# Open the landsea mask and create a combined mask used in the regridding process: 

masks_ds = xr.open_dataset(f'{path_masks}reference_masks.nc')
landsea = xr.where(masks_ds.mask==1, 0, 1)

In [73]:
# Open the datasets and create yearly data: 

#Antarctica: 
ant = xr.open_dataset(data_path + "GRAVIS-3_COSTG_0100_AIS_GRID_TUD_0003.nc")
ant_y = ky.yearlygrace(ant) 

#Greenland: 
gre = xr.open_dataset(data_path + "GRAVIS-3_COSTG_0100_GIS_GRID_TUD_0003.nc")
gre_y = ky.yearlygrace(gre) 


In [74]:
#Regrid the data to match lon lat grid of the sea level equaton. Only regrid the yearly data because we only use that. 

# Use the lon and lat values from the mask directly: Here, points are defined between 2 degrees. It contains for lat all values 89.5 - N*1 with N in N
lon_target = masks_ds['lon'].values
lat_target = masks_ds['lat'].values 

lon2d, lat2d = np.meshgrid(lon_target, lat_target)   #creates lon lat pairs

# CHECK WHY THIS IS EXactly 
ds_out = xr.Dataset(
    {
        "lat": (["lat", "lon"], lat2d),
        "lon": (["lat", "lon"], lon2d)
    }
)

# First make sure nans are filled with zeros. Do that before regridding in stead of after. Otherwise some areas at the boundary are missed and at the boundaries (mainly at antarctica) most mass loss happens. 
ant_y['dm'] = ant_y['dm'].fillna(0)
gre_y['dm'] = gre_y['dm'].fillna(0)

# Create the regridder. Array dataset containin lat lon values. 
regridder = xe.Regridder(ant_y, ds_out, "bilinear", periodic=False, reuse_weights=False, unmapped_to_nan=True)  # Periodic true does not work for some reason in this region... you need apparently full latitude coverage
regridder_gre = xe.Regridder(gre_y, ds_out, "bilinear", periodic=False, reuse_weights=False, unmapped_to_nan=True)

# Regrid dm to lon-lat grid
dm_lonlat = regridder(ant_y["dm"], keep_attrs=True)
dm_lonlat_gre = regridder_gre(gre_y["dm"], keep_attrs=True)
      
dm_lonlat_zero = dm_lonlat.fillna(0) 
dm_lonlat_zero_gre = dm_lonlat_gre.fillna(0) 

In [75]:
# Create a mask of the regridded area of antarctica to make sure all gridded data is masked as land. Combine the ones for greenland and antarctica in one mask called maskje. 

totalant_withnan = dm_lonlat_zero.isel(time=-1) - dm_lonlat_zero.isel(time=0)
totalgre_withnan = dm_lonlat_zero_gre.isel(time=-1) - dm_lonlat_zero_gre.isel(time=0)


# this only works because the values in the totalant are never exactly 0. When this becomes true the code has to be adjusted
maskje = xr.where(
    (np.isnan(totalant_withnan)) | (totalant_withnan == 0),# | np.isnan(totalgre_withnan) | (totalgre_withnan == 0), ## totalant withnan has only nan so I dont need todo this I think
    0,
    1
)

maskjegre = xr.where(
    (np.isnan(totalgre_withnan)) | (totalgre_withnan == 0),
    0,
    1
)


union_mask = xr.where((landsea == 1) |(maskje == 1) | (maskjegre==1), 1, 0)  #(landsea == 1) | 
#union_mask.plot()

In [76]:
# Calculate the total mass loss between 2002 and 2024. 
#totalant = dm_lonlat_zero.isel(time=-1) - dm_lonlat_zero.isel(time=0)

antreference = dm_lonlat_zero - dm_lonlat_zero.isel(time=0) # calculate the values relative to 2002. 

results = []
step=2001
for t in antreference.time:
    step=step+1
    print('Jaar')
    print(step) 
    totalant_t = antreference.sel(time=t)
    sealevel_t = ky.sealevelfunctions(totalant_t, lon_target, lat_target, union_mask)
    results.append(sealevel_t)

# Combine all time slices into a single xarray DataArray
sealevel_all = xr.concat(results, dim=antreference.time)
sealevel_all["time"] = antreference.time


# Make sure that values are nan at land as well



Jaar
2002
Jaar
2003
Jaar
2004
Jaar
2005
Jaar
2006
Jaar
2007
Jaar
2008
Jaar
2009
Jaar
2010
Jaar
2011
Jaar
2012
Jaar
2013
Jaar
2014
Jaar
2015
Jaar
2016
Jaar
2017
Jaar
2018
Jaar
2019
Jaar
2020
Jaar
2021
Jaar
2022
Jaar
2023
Jaar
2024


In [77]:
greenreference = dm_lonlat_zero_gre - dm_lonlat_zero_gre.isel(time=0) 

results_gre = []
step=2001
for t in antreference.time:
    step=step+1
    print('Jaar')
    print(step) 
    totalgre_t = greenreference.sel(time=t)
    sealevel_t_gre = ky.sealevelfunctions(totalgre_t, lon_target, lat_target, union_mask)
    results_gre.append(sealevel_t_gre)

# Combine all time slices into a single xarray DataArray
sealevel_all_gre = xr.concat(results_gre, dim=greenreference.time)
sealevel_all_gre["time"] = greenreference.time


# Make sure that values are nan at land as well



Jaar
2002
Jaar
2003
Jaar
2004
Jaar
2005
Jaar
2006
Jaar
2007
Jaar
2008
Jaar
2009
Jaar
2010
Jaar
2011
Jaar
2012
Jaar
2013
Jaar
2014
Jaar
2015
Jaar
2016
Jaar
2017
Jaar
2018
Jaar
2019
Jaar
2020
Jaar
2021
Jaar
2022
Jaar
2023
Jaar
2024


In [82]:
# Create a dataset with sealevelrise where all values at land are NAN: 

# 1. Create a dataset with the area for every cell on the regirdded grid:

# Get the average radius of the eath [cm] using LMAX (same method as in GRD patterns) 
LMAX = 360

factors = gravtk.units(lmax=LMAX)
rad_e = factors.rad_e/10**5 # Convert to km
rad_e_m = rad_e*1000 #convert to meters

area_da = landsea.copy()

deg2rad = np.pi/180
area_d1 = rad_e_m**2*np.cos(area_da.lat*deg2rad)*deg2rad**2   #calculate for every lat and lon combination the area. actually it depends only on latitude 

area_da.data = area_d1.data[:,np.newaxis]*np.ones(len(area_da.lon))

area_da.name = 'cell area [m**2]'



seamask = (union_mask == 0)
weights_masked = area_da.where(seamask)  ## the masked area data 

#weights_masked.plot()
#plt.show()

sealevel_masked = sealevel_all.where(seamask) 
sealevel_masked_gre = sealevel_all_gre.where(seamask) 


#sealevel_masked.isel(time=-1).plot()
antrisemean = (sealevel_masked * weights_masked).sum(dim=['lat', 'lon'], skipna=True) / weights_masked.sum(dim=['lat', 'lon'], skipna=True)

greenrisemean = (sealevel_masked_gre * weights_masked).sum(dim=['lat', 'lon'], skipna=True) / weights_masked.sum(dim=['lat', 'lon'], skipna=True)

print(antrisemean)
print(greenrisemean)

<xarray.DataArray (time: 23)> Size: 184B
array([0.        , 0.02193252, 0.04769069, 0.05707766, 0.04850043,
       0.0832621 , 0.14956122, 0.14636627, 0.19099912, 0.22402819,
       0.24020636, 0.29663409, 0.35381036, 0.41506793, 0.42105377,
       0.45643142, 0.48705891, 0.53795329, 0.57254203, 0.60218626,
       0.56756072, 0.56146516, 0.59997383])
Coordinates:
  * time     (time) datetime64[ns] 184B 2002-12-31 2003-12-31 ... 2024-12-31
<xarray.DataArray (time: 23)> Size: 184B
array([0.        , 0.01419047, 0.06536791, 0.11373715, 0.17434981,
       0.23735541, 0.30534648, 0.35820974, 0.4413268 , 0.55842545,
       0.65508476, 0.73040662, 0.76763395, 0.80573687, 0.86084522,
       0.87409356, 0.93474934, 1.00476616, 1.10629958, 1.14601504,
       1.18678317, 1.22906018, 1.26920158])
Coordinates:
  * time     (time) datetime64[ns] 184B 2002-12-31 2003-12-31 ... 2024-12-31


In [89]:
# create a nc file with the same format as the data from frederikse to be able to use it easily in the budget 

ant_costg = ky.makefrederiksefile(sealevel_all, antrisemean) 

gre_costg = ky.makefrederiksefile(sealevel_all_gre, greenrisemean)

display(ant_costg)
display(gre_costg)


<xarray.Dataset> Size: 12MB
Dimensions:    (time: 23, lat: 180, lon: 360)
Coordinates:
  * lon        (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * lat        (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * time       (time) int64 184B 2002 2003 2004 2005 ... 2021 2022 2023 2024
Data variables:
    IS_Gravis  (time, lat, lon) float64 12MB nan nan nan ... 0.7223 0.7202
    Meanslr    (time) float64 184B 0.0 0.02193 0.04769 ... 0.5676 0.5615 0.6
Attributes:
    Info:     Based on Gravis data and created using the gravity toolkit

<xarray.Dataset> Size: 12MB
Dimensions:    (time: 23, lat: 180, lon: 360)
Coordinates:
  * lon        (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * lat        (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * time       (time) int64 184B 2002 2003 2004 2005 ... 2021 2022 2023 2024
Data variables:
    IS_Gravis  (time, lat, lon) float64 12MB nan nan nan ... -1.081 -1.099
    Meanslr    (time) float64 184B 0.0 0.01419 0.06537 ... 1.187 1.229 1.269
Attributes:
    Info:     Based on Gravis data and created using the gravity toolkit

In [91]:
ant_costg.to_netcdf(output_path + 'Gravis_Ant_clean.nc')
gre_costg.to_netcdf(output_path + 'Gravis_Gre_clean.nc')

